# 3.6 能不能偷看答案？


[1.3](https://birdhackor.github.io/tiny-perceptron-vlm/1.3.html) 把「貓看狗。」對齊成三個下一字問題，看到貓時應預測看。若一次把整句交給注意力，貓位置就可能讀到右邊的看；它不必學關係，照着答案讀就能有低代價。這會讓訓練看似成功，真正只給開頭生成時卻失敗。我們需要給每個查詢一份「可讀哪些位置」的許可表。

這份表叫遮罩（mask）。因果遮罩（causal mask）規定：第i個查詢可以讀第j個位置，當且僅當j≤i，也就是過去與目前，不能讀未來。對角線j=i是允許的，因為目前字本來就在輸入裡；要預測的是下一字，不是目前字。將被禁止位置的匹配分數設為負無限 `-inf`，softmax後其讀取比例為0，纔不會從它的value帶回資訊。匹配與讀取的前置見 [3.4](https://birdhackor.github.io/tiny-perceptron-vlm/3.4.html)。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.attention import manual_attention

q = torch.zeros(1, 1, 4, 2)
k = torch.zeros_like(q)
v = torch.tensor([[[[1.0, 10.0], [2.0, 20.0], [3.0, 30.0], [4.0, 40.0]]]])
allowed = torch.ones(4, 4, dtype=torch.bool).tril()[None, None]
out, weights = manual_attention(q, k, v, allowed)
changed = v.clone()
changed[:, :, 3] += 100
other, _ = manual_attention(q, k, changed, allowed)
print(weights[0, 0])
print(out[0, 0])
assert torch.allclose(out[:, :, :3], other[:, :, :3])

輸入q、k形狀 `[1,1,4,2]`：一筆、一種讀法、四位置、每位置兩特徵。先只用一種讀法，下一節再增加。它們全0，因此匹配分數相同，允許讀的各位置就平分比例。`torch.bool` 表示真/假許可；`tril()` 只留下主對角線和下面，`[None,None]` 新加兩個長度1的軸，配合資料的筆數與讀法分組。

印出的權重第0列是 `[1,0,0,0]`，第1列 `[0.5,0.5,0,0]`，第2列約 `[1/3,1/3,1/3,0]`，最後各0.25。輸出依序為 `[1,10]`、`[1.5,15]`、`[2,20]`、`[2.5,25]`，可直接按可見範圍平均核對。

下圖每列是一個查詢，四欄由左至右是位置0、1、2、3。綠格允許讀，數字是匹配分數相同時的讀取比例；灰格的×表示禁止，比例為0。對角線也綠。查詢0只有左邊一格可讀，查詢3則四格都可讀，正好對應剛才的權重。右上角灰區是每個位置尚未得到的未來，不是未來永遠不存在。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/foundations_causal.svg")))

程式再把最後一個value增加100。`clone()` 保存原資料不被改；最後的檢查要求前面三個輸出完全不受影響，因為它們無權讀第四格。最後輸出可以改，這正是「防偷看」的可觀察反例，不是隻看矩陣長得像三角形。

練習移除 `tril()`，保留全為True的許可表。先預測每個查詢都平分四位置，改變最後value會影響所有輸出，最後檢查應失敗，再執行核對。shift負責配答案、因果mask負責限制讀資料，兩件事要同時正確。loss指用正確下一字核對預測的猜錯代價；若模型能偷看答案，代價也可能很低，因此還需要這裡的讀取許可與反例檢查。
